In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

# 경로 설정
table_dir = str(PROCESSED_DIR / 'AEGIS-Order-Med1/de')

# 환자별 노이즈 정보
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
patient_mapping_df['new_pt_id'] = patient_mapping_df['new_pt_id'].astype(str)

patient_id_map = patient_mapping_df.set_index('pt_id')['new_pt_id'].to_dict()
date_offset_map = patient_mapping_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_offset_map = patient_mapping_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(table_dir, '**/*.csv'), recursive = True)
print(f"[INFO] 총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# 컬럼 삭제
print("\n[INFO] 컬럼 제거")

columns_to_drop = [
    'atc1_cd'
]

for file in csv_files:
    try:
        medication_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        total_cols_to_remove = len(columns_to_drop)
        present_columns = [col for col in columns_to_drop if col in medication_orders_df.columns]
        removed_cols_count = len(present_columns)

        if removed_cols_count > 0:
            medication_orders_df.drop(columns = present_columns, inplace = True)

        medication_orders_df.to_csv(file, index=False, encoding = 'utf-8')

        print(f"[INFO] {file} - 제거 대상: {total_cols_to_remove}개, 실제 제거: {removed_cols_count}개")

    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# ① pt_id 치환
print("\n[INFO] ① pt_id 치환 시작")
for file in csv_files:
    try:
        medication_orders_df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')

        if 'pt_id' in medication_orders_df.columns:
            medication_orders_df['pt_id'] = medication_orders_df['pt_id'].map(patient_id_map).fillna(medication_orders_df['pt_id'])

        print(medication_orders_df.head(3))

        medication_orders_df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"[INFO] pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] pt_id 오류: {file} → {e}")


In [ ]:
# ② hos_dt ±7일 ±30분
#print("\n② hos_dt 노이즈 적용")
#for file in csv_files:
#    try:
#        medication_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

#        if 'hos_dt' in df.columns:
#            medication_orders_df['hos_dt'] = pd.to_datetime(medication_orders_df['hos_dt'], errors='coerce')
#            medication_orders_df['hos_dt'] = medication_orders_df['hos_dt'] + medication_orders_df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))

#            print(df.head(3))

#            df.to_csv(file, index=False, encoding = 'utf-8')
#            print(f"hos_dt 저장 완료: {file}")
#    except Exception as e:
#        print(f"[hos_dt 오류] {file} → {e}")


In [ ]:
# order_date ±7일 ±30분
print("\n[INFO] order_date 노이즈 적용")
for file in csv_files:
    try:
        medication_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'order_date' in medication_orders_df.columns:
            medication_orders_df['order_date'] = pd.to_datetime(medication_orders_df['order_date'], errors='coerce')
            medication_orders_df['order_date'] = medication_orders_df['order_date'] + medication_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(medication_orders_df.head(3))

            medication_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] order_date 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] order_date 오류: {file} → {e}")


In [ ]:
# order_regi_dt ±7일 ±30분
print("\n[INFO] order_regi_dt 노이즈 적용")
for file in csv_files:
    try:
        medication_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'order_regi_dt' in medication_orders_df.columns:
            medication_orders_df['order_regi_dt'] = pd.to_datetime(medication_orders_df['order_regi_dt'], errors='coerce')
            medication_orders_df['order_regi_dt'] = medication_orders_df['order_regi_dt'] + medication_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(medication_orders_df.head(3))

            medication_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] order_regi_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] order_regi_dt 오류: {file} → {e}")


In [ ]:
# tx_dt ±7일 ±30분
print("\n[INFO] tx_dt 노이즈 적용")
for file in csv_files:
    try:
        medication_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'tx_dt' in medication_orders_df.columns:
            medication_orders_df['tx_dt'] = pd.to_datetime(medication_orders_df['tx_dt'], errors='coerce')
            medication_orders_df['tx_dt'] = medication_orders_df['tx_dt'] + medication_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(medication_orders_df.head(3))

            medication_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] tx_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] tx_dt 오류: {file} → {e}")


In [ ]:
# dc_order_dt ±7일 ±30분
print("\n[INFO] dc_order_dt 노이즈 적용")
for file in csv_files:
    try:
        medication_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'dc_order_dt' in medication_orders_df.columns:
            medication_orders_df['dc_order_dt'] = pd.to_datetime(medication_orders_df['dc_order_dt'], errors='coerce')
            medication_orders_df['dc_order_dt'] = medication_orders_df['dc_order_dt'] + medication_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(medication_orders_df.head(3))

            medication_orders_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] dc_order_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] dc_order_dt 오류: {file} → {e}")


In [ ]:
# 250924 추가
# order_check_dt ±7일 ±30분
print("\n[INFO] order_check_dt 노이즈 적용")
for file in csv_files:
    try:
        medication_orders_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'order_check_dt' in medication_orders_df.columns:
            medication_orders_df['order_check_dt'] = pd.to_datetime(medication_orders_df['order_check_dt'], errors='coerce')
            medication_orders_df['order_check_dt'] = medication_orders_df['order_check_dt'] + medication_orders_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(medication_orders_df.head(3))

            medication_orders_df.to_csv(file, index=False, encoding = 'utf-8-sig')
            print(f"[INFO] order_check_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] order_check_dt 오류: {file} → {e}")
